# 01t. TrailDiscover로 이벤트 라벨 검증 · eventName 기반 라벨

[TrailDiscover](https://github.com/adanalvarez/TrailDiscover)는 CloudTrail **eventName마다** 공격자가 어떻게 쓰는지와 MITRE ATT&CK 전술·기법을 정리한 공개 사전이다 (`events/{서비스}/{eventName}.json`, 467개).

지금 이벤트 라벨(`01_build_labels.ipynb` → `event_labels.csv`)은 **공격 시나리오 기준**이다: Stratus는 파일명(공격 ID), Splunk는 데이터셋 기법, BOTS v3는 주체·IP 규칙. 같은 시나리오 파일 안 이벤트는 API가 무엇이든 그 시나리오의 기법을 받는다.

이 노트북은
1. **검증**: 공격 이벤트마다 우리 기법·전술과 TrailDiscover의 eventName 기법·전술을 비교하고, 맞지 않는 곳을 (출처, 시나리오, eventName) 단위로 기록한다 → `labels/traildiscover_check.md`, `labels/traildiscover_compare.csv`
2. **eventName 기반 라벨**: 공격 이벤트의 기법·전술을 TrailDiscover로 다시 붙인 라벨을 **따로** 만든다 → `data/interim/event_labels_td.csv`. 기존 `event_labels.csv`와 그 뒤 단계는 건드리지 않는다

**공격·정상 구분은 바꾸지 않는다.** TrailDiscover는 "이 API가 공격에 쓰일 수 있다"는 사전이라, 같은 API가 정상 작업에도 쓰인다 (예: `GetCallerIdentity`). 공격 여부는 기존 시나리오 규칙 그대로 쓰고, 기법·전술만 eventName으로 다시 붙인다.

**MITRE 버전 맞추기**: TrailDiscover는 v19 이전 ID를 쓴다. 우리 표(`labels/labels.csv`, v19)로 옮기는 규칙
| TrailDiscover | v19 (labels.csv) | 근거 |
|---|---|---|
| T1562.001 Disable or Modify Tools | T1685 Disable or Modify Tools | 이름 같음 |
| T1562.008 Disable or Modify Cloud Logs | T1685.002 Disable or Modify Cloud Log | 이름 같음 |
| T1562.007 Disable or Modify Cloud Firewall | T1686.001 Cloud Firewall | 이름 같음 |
| T1562.004 Disable or Modify System Firewall | T1686 Disable or Modify System Firewall | 이름 같음 |
| TA0005 Defense Evasion | TA0005 Stealth **또는** TA0112 Defense Impairment | v19에서 둘로 나뉨. 비교할 때는 둘 다 일치로 본다. 라벨에서는 기법이 TA0112 아래면 TA0112, 아니면 TA0005 |
| `T1562`, `T1562.006` 등 위 표에 없는 v19 이전 ID와 우리 표에 없는 ID | (없음) | 옮길 곳이 확실하지 않아 라벨에서 뺀다. 목록은 6번 칸에 남김 |

데이터 위치: `data/raw/traildiscover/repo/` (`git clone https://github.com/adanalvarez/TrailDiscover`, 공개 저장소. `data/` 아래라 git에는 올라가지 않음)

## 1. 준비: TrailDiscover 읽기, ID 옮기기

In [1]:
import json, glob, re, collections
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
L = pd.read_csv(ROOT / 'labels' / 'labels.csv', keep_default_na=False).set_index('label_id')

# TrailDiscover는 v19 이전 ID. v19에서 바뀐 ID는 이름이 같은 v19 ID로 옮긴다 (labels.csv 이름으로 확인)
CROSSWALK = {'T1562.001': 'T1685',      # Impair Defenses: Disable or Modify Tools → Disable or Modify Tools
             'T1562.008': 'T1685.002',  # Disable or Modify Cloud Logs → Disable or Modify Cloud Log
             'T1562.007': 'T1686.001',  # Disable or Modify Cloud Firewall → Cloud Firewall
             'T1562.004': 'T1686'}      # Disable or Modify System Firewall → Disable or Modify System Firewall
OLD_DEFENSE_EVASION = 'TA0005'          # v19 이전 Defense Evasion = v19의 TA0005 Stealth + TA0112 Defense Impairment

def tid(x):
    """'T1078.004 - Cloud Accounts', 'T1078.004: Valid Accounts: ...' → 'T1078.004'"""
    m = re.match(r'\s*(T[AS]?\d{4}(?:\.\d{3})?)', x)
    return m.group(1) if m else None

def parents(t):
    if t not in L.index or L.at[t, 'kind'] != 'technique': return set()
    return {t}.union(*[parents(p) for p in L.at[t, 'parent_ids'].split('|')])

def tactics_of(t):
    if t not in L.index: return set()
    if L.at[t, 'kind'] == 'tactic': return {t}
    return set().union(*[tactics_of(p) for p in L.at[t, 'parent_ids'].split('|')])

def load_td():
    rows = {}
    for f in glob.glob(str(ROOT / 'data/raw/traildiscover/repo/events/**/*.json'), recursive=True):
        d = json.load(open(f))
        raw = [tid(x) for x in d['mitreAttackTechniques'] + d['mitreAttackSubTechniques']]
        unv = [tid(x['technique']) for x in d['unverifiedMitreAttackTechniques'] if 'technique' in x]   # 2개는 형식이 다름 (description, link)
        rows[(d['eventSource'], d['eventName'])] = dict(
            td_tactics_raw=sorted({tid(x) for x in d['mitreAttackTactics']}),
            td_tech_raw=sorted(set(filter(None, raw))),
            td_tech=sorted({CROSSWALK.get(t, t) for t in raw if t}),
            td_unverified=sorted({CROSSWALK.get(t, t) for t in unv if t}),
            usedInWild=d['usedInWild'])
    return rows

td = load_td()
by_name = collections.defaultdict(list)
for (s, n) in td: by_name[n].append(s)
print('TrailDiscover 이벤트', len(td), '| 커밋', __import__('subprocess').run(['git', '-C', str(ROOT / 'data/raw/traildiscover/repo'), 'log', '-1', '--format=%h %ad', '--date=short'], capture_output=True, text=True).stdout.strip())

ev = pd.read_csv(ROOT / 'data/interim/event_labels.csv', keep_default_na=False, low_memory=False)
ev['is_attack'] = ev.is_attack.astype(str) == 'True'
def td_key(r):
    if (r.eventSource, r.eventName) in td: return (r.eventSource, r.eventName)
    if len(by_name.get(r.eventName, [])) == 1: return (by_name[r.eventName][0], r.eventName)   # 서비스 이름 표기만 다른 경우
    return None
ev['td_key'] = ev.apply(td_key, axis=1)
att = ev[ev.is_attack].copy()
print(f'이벤트 {len(ev)} | 공격 {len(att)} | 공격 중 TrailDiscover에 있는 eventName {att.td_key.notna().sum()} ({att.td_key.notna().mean():.0%})')

TrailDiscover 이벤트 467 | 커밋 c03df04 2026-11-10


이벤트 28558 | 공격 8695 | 공격 중 TrailDiscover에 있는 eventName 5967 (69%)


## 2. 기법 비교 (공격 이벤트)

우리 기법(`technique`, 가장 구체적인 것 하나)을 TrailDiscover의 검증된 기법(`mitreAttackTechniques` + `mitreAttackSubTechniques`, v19로 옮김)과 비교한다.

| 분류 | 뜻 |
|---|---|
| 일치 | 우리 기법이 TrailDiscover 기법에 있음 |
| 상위·하위 일치 | 우리 기법의 상위 기법이나 하위 기법이 있음 (예: 우리 T1078.004, TD T1078) |
| 미검증 후보에만 있음 | `unverifiedMitreAttackTechniques`(근거가 약한 후보)에만 있음 |
| 불일치 | TrailDiscover에 이 eventName이 있지만 우리 기법과 겹치지 않음 |
| TD 기법 없음 | eventName은 있지만 기법이 비어 있음 |
| TD에 없음 | TrailDiscover에 eventName이 없음 (공격에 잘 안 쓰이는 API로 본 것) |

In [2]:
def tech_cat(r):
    if r.td_key is None: return 'TD에 없음'
    v = td[r.td_key]; V, U, t = set(v['td_tech']), set(v['td_unverified']), r.technique
    if not V and not U: return 'TD 기법 없음'
    if t in V: return '일치'
    if parents(t) & V or any(t in parents(x) for x in V): return '상위·하위 일치'
    if t in U or parents(t) & U: return '미검증 후보에만 있음'
    return '불일치'
att['tech_cat'] = att.apply(tech_cat, axis=1)
ORDER = ['일치', '상위·하위 일치', '미검증 후보에만 있음', '불일치', 'TD 기법 없음', 'TD에 없음']
tab = pd.crosstab(att.source, att.tech_cat).reindex(columns=[c for c in ORDER if c in set(att.tech_cat)], fill_value=0)
tab.loc['합계'] = tab.sum()
tab['일치율(일치+상위하위)'] = ((tab.get('일치', 0) + tab.get('상위·하위 일치', 0)) / tab.sum(1)).round(3)
tab

/tmp/ipykernel_3479/3124476477.py:13: Pandas4Warning: Starting with pandas version 4.0 all arguments of sum will be keyword-only.
  tab['일치율(일치+상위하위)'] = ((tab.get('일치', 0) + tab.get('상위·하위 일치', 0)) / tab.sum(1)).round(3)


tech_cat,일치,상위·하위 일치,미검증 후보에만 있음,불일치,TD에 없음,일치율(일치+상위하위)
source,,,,,,
botsv3,30,591,15,1,0,0.975
splunk,613,28,538,3859,2709,0.083
stratus,37,16,5,234,19,0.170
합계,680,635,558,4094,2728,0.151


## 3. 전술 비교 (공격 이벤트)

우리 전술(`tactics`, 기법에서 구함)과 TrailDiscover 전술(`mitreAttackTactics`). TD의 TA0005(Defense Evasion)는 우리 TA0005·TA0112 어느 쪽과도 일치로 본다.

| 분류 | 뜻 |
|---|---|
| 같음 | 두 집합이 같음 |
| 일부 겹침 | 하나 이상 겹침 |
| 안 겹침 | 겹치는 전술 없음 |

In [3]:
def td_tactics_v19(key, techs=()):
    """TD 전술 → v19. Defense Evasion은 기법이 TA0112 아래면 TA0112, 아니면 TA0005."""
    out = set()
    for t in td[key]['td_tactics_raw']:
        if t == OLD_DEFENSE_EVASION:
            out.add('TA0112' if any('TA0112' in tactics_of(x) for x in techs) else 'TA0005')
        elif t in L.index:
            out.add(t)
    return out

def tac_cat(r):
    if r.td_key is None: return 'TD에 없음'
    ours = set(r.tactics.split('|')) - {''}
    raw = set(td[r.td_key]['td_tactics_raw'])
    tdset = {t for t in raw if t != OLD_DEFENSE_EVASION} | ({'TA0005', 'TA0112'} & ours if OLD_DEFENSE_EVASION in raw else set())
    if OLD_DEFENSE_EVASION in raw and not ({'TA0005', 'TA0112'} & ours): tdset |= {'TA0005'}
    if not tdset: return 'TD 전술 없음'
    if tdset == ours: return '같음'
    return '일부 겹침' if tdset & ours else '안 겹침'
att['tac_cat'] = att.apply(tac_cat, axis=1)
t2 = pd.crosstab(att.source, att.tac_cat); t2.loc['합계'] = t2.sum(); t2

tac_cat,TD에 없음,같음,안 겹침,일부 겹침
source,,,,
botsv3,0,31,15,591
splunk,2709,4657,239,142
stratus,19,76,193,23
합계,2728,4764,447,756


## 4. 맞지 않는 곳: (출처, 시나리오, eventName) 단위

`불일치`·`미검증 후보에만 있음`·`TD에 없음`을 묶어서 이벤트 수가 많은 순으로 본다. 원인 분류는 규칙으로 붙인 **추정**이다.

| 원인 (추정) | 규칙 |
|---|---|
| 시나리오 라벨이 정찰 호출에 붙음 | TD 전술이 Discovery뿐이거나 API가 Describe/List/Get으로 시작하는데, 우리 기법은 정찰(TA0007)이 아님 |
| 보조 호출 | `flags`에 `helper_call` (Stratus의 `AssumeRole`, `GetCallerIdentity`) |
| TD에 없는 API | TrailDiscover에 eventName이 없음 |
| 같은 API, 다른 해석 | 그 밖 (API 하나가 여러 기법에 쓰이는데 TD와 우리가 다른 쪽을 고름) |

In [4]:
READ = ('Describe', 'List', 'Get', 'Lookup', 'Head', 'Search', 'Check', 'BatchGet', 'Scan', 'Query', 'Select')
def cause(r):
    if 'helper_call' in r['flags']: return '보조 호출'   # r.flags는 pandas 속성이라 [] 로 읽음
    if r.td_key is None: return 'TD에 없는 API'
    tdt = set(td[r.td_key]['td_tactics_raw'])
    if 'TA0007' not in tactics_of(r.technique) and (tdt == {'TA0007'} or r.eventName.startswith(READ)): return '시나리오 라벨이 정찰 호출에 붙음'
    return '같은 API, 다른 해석'
bad = att[att.tech_cat.isin(['불일치', '미검증 후보에만 있음', 'TD에 없음'])].copy()
bad['cause'] = bad.apply(cause, axis=1)
grp = (bad.groupby(['source', 'scenario', 'eventSource', 'eventName', 'technique', 'tactics', 'tech_cat', 'tac_cat', 'cause'], dropna=False)
          .size().rename('events').reset_index())
grp['td_techniques'] = grp.apply(lambda r: '|'.join(td[td_key(r)]['td_tech']) if td_key(r) else '', axis=1)
grp['td_unverified'] = grp.apply(lambda r: '|'.join(td[td_key(r)]['td_unverified']) if td_key(r) else '', axis=1)
grp['td_tactics'] = grp.apply(lambda r: '|'.join(td[td_key(r)]['td_tactics_raw']) if td_key(r) else '', axis=1)
grp = grp.sort_values('events', ascending=False).reset_index(drop=True)
print('맞지 않는 공격 이벤트', len(bad), '| 묶음', len(grp))
print(grp.groupby('cause').events.agg(['count', 'sum']).rename(columns={'count': '묶음', 'sum': '이벤트'}).sort_values('이벤트', ascending=False).to_string())
grp.head(25)

맞지 않는 공격 이벤트 7380 | 묶음 242
                     묶음   이벤트
cause                        
같은 API, 다른 해석        77  4250
TD에 없는 API          143  2728
시나리오 라벨이 정찰 호출에 붙음   15   387
보조 호출                 7    15


,source,scenario,eventSource,eventName,technique,tactics,tech_cat,tac_cat,cause,events,td_techniques,td_unverified,td_tactics
0,splunk,aws_iam_excessive_list_command_usage,s3.amazonaws.com,ListObjects,T1580,TA0007,불일치,같음,"같은 API, 다른 해석",2703,T1619,T1083|T1213,TA0007
1,splunk,aws_cloudwatch_logs_high_volume_retrieval,logs.amazonaws.com,GetLogEvents,T1530,TA0009,TD에 없음,TD에 없음,TD에 없는 API,650,,,
2,splunk,aws_iam_excessive_list_command_usage,iam.amazonaws.com,ListRoles,T1580,TA0007,불일치,같음,"같은 API, 다른 해석",555,T1087,T1069|T1518,TA0007
3,splunk,aws_iam_delete_policy,iam.amazonaws.com,DeletePolicy,T1098,TA0003|TA0004,TD에 없음,TD에 없음,TD에 없는 API,110,,,
4,splunk,aws_exfil_high_no_getobject,s3.amazonaws.com,GetObject,T1530,TA0009,미검증 후보에만 있음,안 겹침,시나리오 라벨이 정찰 호출에 붙음,100,T1048,T1071|T1074|T1530|T1570,TA0010
5,splunk,aws_iam_successful_group_deletion,iam.amazonaws.com,DeleteGroup,T1098,TA0003|TA0004,TD에 없음,TD에 없음,TD에 없는 API,85,,,
6,splunk,aws_iam_excessive_list_command_usage,route53.amazonaws.com,ListHostedZonesByVPC,T1580,TA0007,TD에 없음,TD에 없음,TD에 없는 API,66,,,
7,splunk,aws_iam_accessdenied_discovery_events,elasticloadbalancing.amazonaws.com,DescribeLoadBalancers,T1580,TA0007,미검증 후보에만 있음,같음,"같은 API, 다른 해석",64,T1526,T1046|T1133|T1482|T1580|T1590,TA0007
8,splunk,aws_getpassworddata,ec2.amazonaws.com,GetPasswordData,T1552,TA0006,불일치,같음,시나리오 라벨이 정찰 호출에 붙음,60,T1555,T1070|T1078|T1098|T1531|T1548.002,TA0006
9,splunk,aws_createaccesskey,iam.amazonaws.com,CreateAccessKey,T1098.001,TA0003|TA0004,미검증 후보에만 있음,같음,"같은 API, 다른 해석",54,T1078|T1078.004|T1136|T1136.003,T1098,TA0003|TA0004


## 5. 기록 저장

- `labels/traildiscover_compare.csv`: 공격 이벤트를 (출처, 시나리오, eventName, 우리 기법) 단위로 묶은 비교 전체 (일치 포함)
- `labels/traildiscover_check.md`: 요약과 맞지 않는 곳 목록

In [5]:
full = (att.groupby(['source', 'scenario', 'eventSource', 'eventName', 'technique', 'tactics', 'tech_cat', 'tac_cat'], dropna=False)
           .size().rename('events').reset_index())
full['td_techniques'] = full.apply(lambda r: '|'.join(td[td_key(r)]['td_tech']) if td_key(r) else '', axis=1)
full['td_unverified'] = full.apply(lambda r: '|'.join(td[td_key(r)]['td_unverified']) if td_key(r) else '', axis=1)
full['td_tactics'] = full.apply(lambda r: '|'.join(td[td_key(r)]['td_tactics_raw']) if td_key(r) else '', axis=1)
full = full.merge(grp[['source', 'scenario', 'eventName', 'technique', 'cause']], how='left', on=['source', 'scenario', 'eventName', 'technique'])
full.sort_values(['source', 'scenario', 'events'], ascending=[True, True, False]).to_csv(ROOT / 'labels/traildiscover_compare.csv', index=False)

def mdtable(df):
    return '\n'.join(['| ' + ' | '.join(map(str, df.columns)) + ' |', '|' + '---|' * len(df.columns)] +
                     ['| ' + ' | '.join(str(x).replace('|', ', ') for x in r) + ' |' for r in df.itertuples(index=False)])
c = grp.groupby('cause').events.agg(['count', 'sum']).rename(columns={'count': '묶음', 'sum': '이벤트'}).sort_values('이벤트', ascending=False).reset_index()
lines = ['# TrailDiscover 검증 기록', '',
         f'`notebooks/01_traildiscover_labels.ipynb`가 만든다. 공격 이벤트 {len(att)}개의 기법·전술(시나리오 기준)을 TrailDiscover의 eventName 기법·전술과 비교했다.', '',
         '## 기법 비교 (출처별 이벤트 수)', '', mdtable(tab.reset_index()), '',
         '## 전술 비교', '', mdtable(t2.reset_index()), '',
         '## 맞지 않는 곳 원인 (추정)', '', mdtable(c), '',
         '## 맞지 않는 곳 전체 (이벤트 수 순)', '',
         '`tech_cat`: 불일치 / 미검증 후보에만 있음 / TD에 없음. `td_*`: TrailDiscover 값 (기법은 v19로 옮김).', '',
         mdtable(grp[['events', 'source', 'scenario', 'eventName', 'technique', 'td_techniques', 'td_unverified', 'tactics', 'td_tactics', 'tech_cat', 'cause']])]
(ROOT / 'labels/traildiscover_check.md').write_text('\n'.join(lines) + '\n')
print('저장: labels/traildiscover_compare.csv', len(full), '행 | labels/traildiscover_check.md')

저장: labels/traildiscover_compare.csv 334 행 | labels/traildiscover_check.md


## 6. eventName 기반 라벨 (TrailDiscover)

공격 이벤트의 기법·전술을 TrailDiscover로 다시 붙인다. 공격·정상 구분과 정상 이벤트는 그대로.

| 단계 | 규칙 |
|---|---|
| 기법 | TD 검증 기법(v19로 옮김) 중 `labels.csv`에 있는 것 **전부** (멀티라벨) + 상위 기법 (L3) |
| 전술 | TD 전술(v19로 옮김) ∪ 기법들의 전술 (L2: 기법의 전술은 늘 포함) |
| TD에 없거나 쓸 수 있는 기법이 없음 | **시나리오 라벨 유지** (`label_source = scenario_fallback`) |
| `technique` 열 | 기법 중 가장 구체적인 것 하나 (하위 기법 우선, 같은 수준이면 ID 순). 압축·분석용 |
| `label_source` | `traildiscover` / `scenario_fallback` / 정상은 기존 값 |

`labels.csv`에 없어서 뺀 TD 기법은 아래에 목록으로 남긴다 (라벨 체계에 추가할지 정할 때).

In [6]:
dropped = collections.Counter()
def td_label(r):
    if not r.is_attack:
        return r.techniques, r.tactics, r.technique, r.label_source
    if r.td_key is not None:
        raw = td[r.td_key]['td_tech']
        keep = [t for t in raw if t in L.index and L.at[t, 'kind'] == 'technique']
        dropped.update(t for t in raw if t not in keep)
        if keep:
            techs = set().union(*[parents(t) for t in keep])
            tacs = td_tactics_v19(r.td_key, techs) | set().union(*[tactics_of(t) for t in techs])
            main = sorted(keep, key=lambda t: (-t.count('.'), t))[0]
            return '|'.join(sorted(techs)), '|'.join(sorted(tacs)), main, 'traildiscover'
    return r.techniques, r.tactics, r.technique, 'scenario_fallback'

new = ev.copy()
new[['techniques', 'tactics', 'technique', 'label_source']] = pd.DataFrame(new.apply(td_label, axis=1).tolist(), index=new.index)
new['is_attack'] = new.is_attack.map({True: 'True', False: 'False'})
new.drop(columns=['td_key']).to_csv(ROOT / 'data/interim/event_labels_td.csv', index=False)

a_new = new[new.is_attack == 'True']
print('저장: data/interim/event_labels_td.csv', len(new))
print('공격 라벨 출처:', a_new.label_source.value_counts().to_dict())
print('공격 이벤트 중 기법이 바뀐 수:', int((a_new.techniques != ev.loc[a_new.index, 'techniques']).sum()),
      '| 전술이 바뀐 수:', int((a_new.tactics != ev.loc[a_new.index, 'tactics']).sum()))
print('labels.csv에 없어서 뺀 TD 기법 (이벤트 수):', dict(dropped.most_common()))

저장: data/interim/event_labels_td.csv 28558
공격 라벨 출처: {'traildiscover': 5964, 'scenario_fallback': 2731}
공격 이벤트 중 기법이 바뀐 수: 5326 | 전술이 바뀐 수: 1217
labels.csv에 없어서 뺀 TD 기법 (이벤트 수): {'T1562': 25, 'T1021.004': 19, 'T1133': 4, 'T1078.001': 4, 'T1499': 2, 'T1550': 2, 'T1556': 2, 'T1070': 2, 'T1021.007': 1, 'T1136.001': 1, 'T1119': 1}


## 7. 라벨 분포 비교 (공격 이벤트, 이벤트 하나에 라벨 여러 개)

In [7]:
def dist(df, col):
    return df[df.is_attack.astype(str) == 'True'][col].str.split('|').explode().value_counts()
cmp_tac = pd.DataFrame({'시나리오 기준': dist(ev, 'tactics'), 'TrailDiscover': dist(new, 'tactics')}).fillna(0).astype(int)
cmp_tac.index = [f'{i} {L.at[i, "name"]}' if i in L.index else i for i in cmp_tac.index]
cmp_tech = pd.DataFrame({'시나리오 기준': dist(ev, 'techniques'), 'TrailDiscover': dist(new, 'techniques')}).fillna(0).astype(int)
print('기법 종류: 시나리오 기준', (cmp_tech['시나리오 기준'] > 0).sum(), '→ TrailDiscover', (cmp_tech['TrailDiscover'] > 0).sum())
display(cmp_tac.sort_values('시나리오 기준', ascending=False))
cmp_tech.sort_values('TrailDiscover', ascending=False).head(40)

기법 종류: 시나리오 기준 36 → TrailDiscover 35


,시나리오 기준,TrailDiscover
TA0007 Discovery,6463,6581
TA0009 Collection,759,651
TA0040 Impact,616,616
TA0003 Persistence,406,1012
TA0004 Privilege Escalation,403,1008
TA0006 Credential Access,245,289
TA0002 Execution,87,12
TA0112 Defense Impairment,85,37
TA0001 Initial Access,79,173
TA0005 Stealth,79,145


,시나리오 기준,TrailDiscover
techniques,,
T1619,0,2719
T1580,5391,2372
T1526,1071,899
T1098,323,867
T1530,759,650
T1021,19,603
T1496,609,599
T1087,1,591
T1555,49,209
